In [1]:
from pyspark.sql import SparkSession
from datetime import datetime
from pyspark.sql.types import StructField, StructType, IntegerType, TimestampType, DecimalType, StringType
from decimal import Decimal

spark = SparkSession.builder.appName("interview_prep").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/03 11:49:22 WARN Utils: Your hostname, codespaces-9ec455, resolves to a loopback address: 127.0.0.1; using 10.0.3.193 instead (on interface eth0)
26/07/03 11:49:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 11:49:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
"""
A bank's fraud team wants to flag accounts where 3 or more debit transactions of $5,000 or more occurred within any rolling 1-hour window.
Table: transactions
columntypenotesaccount_idINTtxn_timestampTIMESTAMPamountDECIMAL(12,2)negative = debit, positive = credittxn_typeVARCHARATM, WIRE, DEPOSIT, etc.
Task: Return the account_id and the txn_timestamp of the first transaction in the earliest qualifying 1-hour window for that account. One row per flagged account, sorted by account_id.
"""

data = [
    (101, datetime(2024, 3, 1,  9,  0,  0), Decimal("-6000.00"), "ATM"),
    (101, datetime(2024, 3, 1,  9, 25,  0), Decimal("-5500.00"), "WIRE"),
    (101, datetime(2024, 3, 1,  9, 50,  0), Decimal("-7000.00"), "ATM"),
    (101, datetime(2024, 3, 1, 14,  0,  0), Decimal("-8000.00"), "WIRE"),

    (102, datetime(2024, 3, 1, 10,  0,  0), Decimal("-5000.00"), "ATM"),
    (102, datetime(2024, 3, 1, 10, 30,  0), Decimal( "3000.00"), "DEPOSIT"),
    (102, datetime(2024, 3, 1, 10, 45,  0), Decimal("-5200.00"), "ATM"),

    (103, datetime(2024, 3, 1, 11,  0,  0), Decimal("-5000.00"), "ATM"),
    (103, datetime(2024, 3, 1, 11, 30,  0), Decimal("-4999.99"), "ATM"),
    (103, datetime(2024, 3, 1, 11, 45,  0), Decimal("-6000.00"), "WIRE"),

    (104, datetime(2024, 3, 1,  8,  0,  0), Decimal("-5500.00"), "ATM"),
    (104, datetime(2024, 3, 1,  8, 30,  0), Decimal("-6000.00"), "WIRE"),
    (104, datetime(2024, 3, 1,  8, 45,  0), Decimal("-5500.00"), "ATM"),
    (104, datetime(2024, 3, 1, 12,  0,  0), Decimal("-7000.00"), "ATM"),
    (104, datetime(2024, 3, 1, 12, 15,  0), Decimal("-5000.00"), "WIRE"),
    (104, datetime(2024, 3, 1, 12, 50,  0), Decimal("-8000.00"), "ATM"),

    (105, datetime(2024, 3, 1,  9,  0,  0), Decimal("10000.00"), "DEPOSIT"),
    (105, datetime(2024, 3, 1,  9, 20,  0), Decimal( "6000.00"), "DEPOSIT"),
    (105, datetime(2024, 3, 1,  9, 50,  0), Decimal( "8000.00"), "DEPOSIT"),

    (106, datetime(2024, 3, 1, 13,  0,  0), Decimal("-5000.00"), "ATM"),
    (106, datetime(2024, 3, 1, 13, 30,  0), Decimal("-6000.00"), "ATM"),
    (106, datetime(2024, 3, 1, 14,  0,  0), Decimal("-5500.00"), "ATM"),

    (107, datetime(2024, 3, 1, 15,  0,  0), Decimal("-5500.00"), "ATM"),
    (107, datetime(2024, 3, 1, 15, 10,  0), Decimal("-6000.00"), "WIRE"),
    (107, datetime(2024, 3, 1, 15, 20,  0), Decimal("-5500.00"), "ATM"),
    (107, datetime(2024, 3, 1, 15, 30,  0), Decimal("-7000.00"), "ATM"),

    (108, datetime(2024, 3, 1, 16,  0,  0), Decimal("-5500.00"), "ATM"),
    (108, datetime(2024, 3, 1, 16, 30,  0), Decimal("-6000.00"), "ATM"),
    (108, datetime(2024, 3, 1, 17, 30,  0), Decimal("-5500.00"), "ATM"),

    (109, datetime(2024, 3, 2,  9,  0,  0), Decimal("-5500.00"), "ATM"),
    (109, datetime(2024, 3, 2,  9, 30,  0), Decimal("-6000.00"), "WIRE"),
    (109, datetime(2024, 3, 3,  9,  0,  0), Decimal("-7000.00"), "ATM"),
    (109, datetime(2024, 3, 3,  9, 15,  0), Decimal("-5500.00"), "WIRE"),
    (109, datetime(2024, 3, 3,  9, 45,  0), Decimal("-5800.00"), "ATM"),
]

schema = StructType([
    StructField("account_id",    IntegerType(),       False),
    StructField("txn_timestamp", TimestampType(),     False),
    StructField("amount",        DecimalType(12, 2),  False),
    StructField("txn_type",      StringType(),        False),
])

df = spark.createDataFrame(data, schema)

df.createOrReplaceTempView("transactions")

spark.sql("""
          with qualifying as (
            select account_id, txn_timestamp
            from transactions
            where amount <= -5000
          ),
          with_lag as (
            select account_id, txn_timestamp as third_ts,
            lag(txn_timestamp, 2) over (partition by account_id order by txn_timestamp) as first_ts
            from qualifying
          )
          select account_id,
          min(first_ts) as first_txn_timestamp
          from with_lag
          where first_ts is not null
          and (unix_timestamp(third_ts) - unix_timestamp(first_ts)) <= 3600
          group by account_id
          order by account_id
""").show()

+----------+-------------------+
|account_id|first_txn_timestamp|
+----------+-------------------+
|       101|2024-03-01 09:00:00|
|       104|2024-03-01 08:00:00|
|       106|2024-03-01 13:00:00|
|       107|2024-03-01 15:00:00|
|       109|2024-03-03 09:00:00|
+----------+-------------------+



In [3]:
"""
A bank wants to identify, for each branch, the top 3 borrowers by total loan amount. The result will be used by relationship managers for client retention outreach.
Sorted by branch_id, then rank_in_branch.
"""

from pyspark.sql import SparkSession
from datetime import datetime
from datetime import date
from pyspark.sql.types import StructField, StructType, IntegerType, TimestampType, DecimalType, StringType, DateType
from decimal import Decimal

loans_data = [
    # Branch 1: customer A has 2 loans, B and C tie at #2
    (1001, 1, 501, Decimal("250000.00"), date(2023,  1, 15)),
    (1002, 1, 501, Decimal("250000.00"), date(2023,  6, 10)),
    (1003, 1, 502, Decimal("300000.00"), date(2022,  5,  1)),
    (1004, 1, 503, Decimal("300000.00"), date(2023,  3, 20)),
    (1005, 1, 504, Decimal("200000.00"), date(2023,  8, 12)),

    # Branch 2: only 2 customers — what happens to "top 3"?
    (1006, 2, 601, Decimal("500000.00"), date(2023,  2,  5)),
    (1007, 2, 602, Decimal("400000.00"), date(2023,  4, 18)),

    # Branch 3: clean separation, no ties
    (1008, 3, 701, Decimal("1000000.00"), date(2023,  1, 10)),
    (1009, 3, 702, Decimal( "800000.00"), date(2023,  2, 22)),
    (1010, 3, 703, Decimal( "600000.00"), date(2023,  3, 14)),
    (1011, 3, 704, Decimal( "400000.00"), date(2023,  5,  8)),

    # Branch 4: THREE-WAY TIE at the top
    (1012, 4, 801, Decimal("500000.00"), date(2022, 11, 30)),
    (1013, 4, 802, Decimal("500000.00"), date(2023,  1, 25)),
    (1014, 4, 803, Decimal("500000.00"), date(2023,  7, 14)),
    (1015, 4, 804, Decimal("300000.00"), date(2023,  9,  9)),

    # Branch 5: same customer with multiple loans across years
    (1016, 5, 901, Decimal("100000.00"), date(2020,  3,  1)),
    (1017, 5, 901, Decimal("150000.00"), date(2021,  3,  1)),
    (1018, 5, 901, Decimal("200000.00"), date(2022,  3,  1)),
    (1019, 5, 902, Decimal("400000.00"), date(2023,  4,  1)),
    (1020, 5, 903, Decimal("350000.00"), date(2023,  5,  1)),
]

loans_schema = StructType([
    StructField("loan_id",     IntegerType(),       False),
    StructField("branch_id",   IntegerType(),       False),
    StructField("customer_id", IntegerType(),       False),
    StructField("loan_amount", DecimalType(12, 2),  False),
    StructField("loan_date",   DateType(),          False),
])

loans = spark.createDataFrame(loans_data, loans_schema)
loans.createOrReplaceTempView("loans")

spark.sql("""
           WITH GROUPED AS(
                SELECT branch_id, customer_id, SUM(loan_amount) AS total_loan, DENSE_RANK() OVER (PARTITION BY branch_id ORDER BY SUM(loan_amount) DESC, MIN(loan_date) ASC) AS rank
                FROM LOANS
                GROUP BY branch_id, customer_id
           )
           SELECT branch_id, customer_id, total_loan AS total_loan_amount, rank AS rank_in_branch
           FROM grouped
           WHERE rank <= 3
           ORDER BY branch_id, rank_in_branch
           """).show()

+---------+-----------+-----------------+--------------+
|branch_id|customer_id|total_loan_amount|rank_in_branch|
+---------+-----------+-----------------+--------------+
|        1|        501|        500000.00|             1|
|        1|        502|        300000.00|             2|
|        1|        503|        300000.00|             3|
|        2|        601|        500000.00|             1|
|        2|        602|        400000.00|             2|
|        3|        701|       1000000.00|             1|
|        3|        702|        800000.00|             2|
|        3|        703|        600000.00|             3|
|        4|        801|        500000.00|             1|
|        4|        802|        500000.00|             2|
|        4|        803|        500000.00|             3|
|        5|        901|        450000.00|             1|
|        5|        902|        400000.00|             2|
|        5|        903|        350000.00|             3|
+---------+-----------+--------

In [4]:
from datetime import date
from decimal import Decimal
from pyspark.sql.types import StructType, StructField, IntegerType, DecimalType, DateType

balance_data = [
    # 201: multiple streaks — Jan-Mar (3), May-Jun (2), Aug (1)
    (201, date(2023,  1, 1), Decimal( "-500.00")),
    (201, date(2023,  2, 1), Decimal( "-200.00")),
    (201, date(2023,  3, 1), Decimal( "-100.00")),
    (201, date(2023,  4, 1), Decimal(  "200.00")),
    (201, date(2023,  5, 1), Decimal(  "-50.00")),
    (201, date(2023,  6, 1), Decimal( "-300.00")),
    (201, date(2023,  7, 1), Decimal(  "500.00")),
    (201, date(2023,  8, 1), Decimal( "-100.00")),
    (201, date(2023,  9, 1), Decimal(  "400.00")),

    # 202: never overdrawn — should produce ZERO rows
    (202, date(2023,  1, 1), Decimal( "1000.00")),
    (202, date(2023,  2, 1), Decimal(  "800.00")),
    (202, date(2023,  3, 1), Decimal( "1200.00")),
    (202, date(2023,  4, 1), Decimal(  "900.00")),

    # 203: ALWAYS overdrawn — one big 12-month streak
    (203, date(2023,  1, 1), Decimal( "-100.00")),
    (203, date(2023,  2, 1), Decimal( "-150.00")),
    (203, date(2023,  3, 1), Decimal( "-200.00")),
    (203, date(2023,  4, 1), Decimal( "-250.00")),
    (203, date(2023,  5, 1), Decimal( "-300.00")),
    (203, date(2023,  6, 1), Decimal( "-350.00")),
    (203, date(2023,  7, 1), Decimal( "-400.00")),
    (203, date(2023,  8, 1), Decimal( "-450.00")),
    (203, date(2023,  9, 1), Decimal( "-500.00")),
    (203, date(2023, 10, 1), Decimal( "-550.00")),
    (203, date(2023, 11, 1), Decimal( "-600.00")),
    (203, date(2023, 12, 1), Decimal( "-650.00")),

    # 204: alternating — six 1-month streaks
    (204, date(2023,  1, 1), Decimal( "-100.00")),
    (204, date(2023,  2, 1), Decimal(  "100.00")),
    (204, date(2023,  3, 1), Decimal( "-100.00")),
    (204, date(2023,  4, 1), Decimal(  "100.00")),
    (204, date(2023,  5, 1), Decimal( "-100.00")),
    (204, date(2023,  6, 1), Decimal(  "100.00")),
    (204, date(2023,  7, 1), Decimal( "-100.00")),
    (204, date(2023,  8, 1), Decimal(  "100.00")),
    (204, date(2023,  9, 1), Decimal( "-100.00")),
    (204, date(2023, 10, 1), Decimal(  "100.00")),
    (204, date(2023, 11, 1), Decimal( "-100.00")),

    # 205: EDGE CASE — gap in the data (no March entry)
    (205, date(2023,  1, 1), Decimal( "-100.00")),
    (205, date(2023,  2, 1), Decimal( "-100.00")),
    # March 2023 is missing from the data entirely
    (205, date(2023,  4, 1), Decimal( "-100.00")),
    (205, date(2023,  5, 1), Decimal( "-100.00")),
    (205, date(2023,  6, 1), Decimal(  "200.00")),

    # 206: zero balance — does it count? Decide and defend.
    (206, date(2023,  1, 1), Decimal( "-100.00")),
    (206, date(2023,  2, 1), Decimal(    "0.00")),
    (206, date(2023,  3, 1), Decimal( "-100.00")),
]

balance_schema = StructType([
    StructField("account_id",           IntegerType(),       False),
    StructField("balance_month",        DateType(),          False),
    StructField("end_of_month_balance", DecimalType(12, 2),  False),
])

monthly_balance = spark.createDataFrame(balance_data, balance_schema)
monthly_balance.createOrReplaceTempView("monthly_balance")

spark.sql("""
    WITH qualifying AS(
        SELECT account_id, balance_month, end_of_month_balance, ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY balance_month) AS rn
        FROM  monthly_balance
        WHERE end_of_month_balance < 0
    ),
    anchored AS (
        SELECT account_id, balance_month, add_months(balance_month, -CAST(rn AS INT)) AS streak_anchor
        FROM qualifying
    )
    SELECT account_id, MIN(balance_month) AS streak_start_month, MAX(balance_month) AS streak_end_month, COUNT(*) AS num_months
    FROM anchored
    GROUP BY account_id, streak_anchor
    ORDER BY account_id, streak_start_month
    -- SELECT * FROM anchored

""").show()

+----------+------------------+----------------+----------+
|account_id|streak_start_month|streak_end_month|num_months|
+----------+------------------+----------------+----------+
|       201|        2023-01-01|      2023-03-01|         3|
|       201|        2023-05-01|      2023-06-01|         2|
|       201|        2023-08-01|      2023-08-01|         1|
|       203|        2023-01-01|      2023-12-01|        12|
|       204|        2023-01-01|      2023-01-01|         1|
|       204|        2023-03-01|      2023-03-01|         1|
|       204|        2023-05-01|      2023-05-01|         1|
|       204|        2023-07-01|      2023-07-01|         1|
|       204|        2023-09-01|      2023-09-01|         1|
|       204|        2023-11-01|      2023-11-01|         1|
|       205|        2023-01-01|      2023-02-01|         2|
|       205|        2023-04-01|      2023-05-01|         2|
|       206|        2023-01-01|      2023-01-01|         1|
|       206|        2023-03-01|      202

In [5]:
"""
Given an array of strings, group the anagrams together. You can return the answer in any order.
An anagram is a word formed by rearranging the letters of another — e.g., "eat" and "tea".
"""

from typing import List
from collections import defaultdict

def group_anagrams(strs: List[str]) -> List[List[str]]:
    # your code here
    anagram_dict = defaultdict(list)
    for i in range(len(strs)):
        anagram_dict[''.join(sorted(strs[i]))].append(strs[i])
    # print(anagram_dict)
    return list(anagram_dict.values())
    pass

# Test
print(group_anagrams(["eat","tea","tan","ate","nat","bat"]))
print(group_anagrams([""]))
print(group_anagrams(["a"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
[['']]
[['a']]


In [6]:
"""
Given an integer array nums and an integer k, return the k most frequent elements. You may return the answer in any order.
"""

from typing import List
from collections import Counter

def top_k_frequent(nums: List[int], k: int) -> List[int]:
    # your code here
    # count_dict = defaultdict(int)
    count_dict = Counter(nums)
    count_dict = dict(sorted(count_dict.items(), key = lambda x: x[1], reverse = True))
    return list(count_dict.keys())[:k]
    pass

# Test
print(top_k_frequent([1,1,1,2,2,3], 2))
print(top_k_frequent([1], 1))
print(top_k_frequent([4,4,4,5,5,6,6,6,6,7], 3))

[1, 2]
[1]
[6, 4, 5]


In [7]:
"""

"""

from typing import List
from collections import defaultdict

def longest_normal_burst(merchants: List[int], k: int) -> int:
    # your code here
    # count, res_list = 0, list()
    # for i in range(len(merchants)):
    #     if len(set(res_list)) > k:
    #         res_list = res_list[1:]
    #     res_list.append(merchants[i])
    #     if len(set(res_list)) <= k:
    #         count = max(count, len(res_list))
    # return count
    # left, right = 0, 1
    # count = 0
    # res_list = list()
    # while left < right and right <= len(merchants):
    #     res_list = merchants[left : right]
    #     # print(res_list)
    #     if len(set(res_list)) > k:
    #         left += 1
    #     if len(set(res_list)) <= k:
    #         count = max(count, len(res_list))
    #     right += 1
    # return count

    count_dict = defaultdict(int)
    left, max_len = 0, 0
    for right in range(len(merchants)):
        count_dict[merchants[right]] += 1
        while len(count_dict) > k:
            count_dict[merchants[left]] -= 1
            if count_dict[merchants[left]] == 0:
                del count_dict[merchants[left]]
            left += 1
        max_len = max(max_len, right - left + 1)
    return max_len
        
    pass

# Tests
print(longest_normal_burst([1, 2, 1, 2, 3], 2))         # 4
print(longest_normal_burst([1, 2, 1, 3, 4], 3))         # 4
print(longest_normal_burst([5, 5, 5, 5], 1))            # 4
print(longest_normal_burst([1, 2, 3], 5))               # 3

4
4
4
3


In [8]:
"""
A bank's CFO wants a simple report: for each branch, the month-over-month percentage change in total deposits.
"""

from datetime import date
from decimal import Decimal
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DecimalType

deposits_data = [
    # Branch 1: clean monthly progression
    (1, date(2024, 1, 5),  Decimal("10000.00")),
    (1, date(2024, 1, 20), Decimal( "5000.00")),
    (1, date(2024, 2, 8),  Decimal("12000.00")),
    (1, date(2024, 2, 15), Decimal( "6000.00")),
    (1, date(2024, 3, 3),  Decimal( "9000.00")),
    (1, date(2024, 3, 22), Decimal( "9000.00")),

    # Branch 2: one month, then drop, then recover
    (2, date(2024, 1, 10), Decimal("20000.00")),
    (2, date(2024, 2, 14), Decimal("15000.00")),
    (2, date(2024, 3, 5),  Decimal("25000.00")),

    # Branch 3: only one month of data
    (3, date(2024, 1, 18), Decimal( "8000.00")),

    # Branch 4: gap month (Feb missing)
    (4, date(2024, 1, 7),  Decimal("11000.00")),
    (4, date(2024, 3, 9),  Decimal("14000.00")),
]

deposits_schema = StructType([
    StructField("branch_id",    IntegerType(),       False),
    StructField("deposit_date", DateType(),          False),
    StructField("amount",       DecimalType(12, 2),  False),
])

deposits = spark.createDataFrame(deposits_data, deposits_schema)
deposits.createOrReplaceTempView("deposits")
spark.sql("""
    WITH first_cte AS(
        SELECT branch_id, amount, deposit_date, trunc(deposit_date, 'MM') AS first_date
        FROM deposits
    )
    SELECT branch_id, first_date AS month, SUM(amount) AS total_deposit, LAG(SUM(amount)) OVER (PARTITION BY branch_id ORDER BY first_date) AS prev_month_deposits,
          ROUND((SUM(amount) / LAG(SUM(amount)) OVER (PARTITION BY branch_id ORDER BY first_date) - 1) * 100, 2) AS mom_pct_change
    FROM first_cte
    GROUP BY branch_id, first_date
    ORDER BY branch_id, first_date
""").show()

+---------+----------+-------------+-------------------+--------------+
|branch_id|     month|total_deposit|prev_month_deposits|mom_pct_change|
+---------+----------+-------------+-------------------+--------------+
|        1|2024-01-01|     15000.00|               NULL|          NULL|
|        1|2024-02-01|     18000.00|           15000.00|         20.00|
|        1|2024-03-01|     18000.00|           18000.00|          0.00|
|        2|2024-01-01|     20000.00|               NULL|          NULL|
|        2|2024-02-01|     15000.00|           20000.00|        -25.00|
|        2|2024-03-01|     25000.00|           15000.00|         66.67|
|        3|2024-01-01|      8000.00|               NULL|          NULL|
|        4|2024-01-01|     11000.00|               NULL|          NULL|
|        4|2024-03-01|     14000.00|           11000.00|         27.27|
+---------+----------+-------------+-------------------+--------------+



In [9]:
"""
A bank's HR is auditing salary structure to identify cases where a direct report earns more than their manager. They want a clean report.
"""

from decimal import Decimal
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType

employees_data = [
    # Top of hierarchy — no manager
    (1,  "Alice Chen",      Decimal("500000.00"), None),

    # VPs reporting to CEO
    (2,  "Bob Martinez",    Decimal("300000.00"), 1),
    (3,  "Carol Singh",     Decimal("280000.00"), 1),

    # Managers reporting to VPs
    (4,  "Dave Kim",        Decimal("150000.00"), 2),
    (5,  "Eve Patel",       Decimal("145000.00"), 2),
    (6,  "Frank Liu",       Decimal("160000.00"), 3),

    # Individual contributors — some anomalies
    (7,  "Grace Wu",        Decimal("175000.00"), 4),  # earns more than Dave (mgr)
    (8,  "Henry Adams",     Decimal( "95000.00"), 4),  # normal
    (9,  "Ivy Jensen",      Decimal("200000.00"), 5),  # earns more than Eve (mgr)
    (10, "Jack Brown",      Decimal( "80000.00"), 5),  # normal
    (11, "Kate Davis",      Decimal("170000.00"), 6),  # earns more than Frank (mgr)
    (12, "Liam Foster",     Decimal( "85000.00"), 6),  # normal

    # Edge case: employee whose manager_id points to no one
    # (handled cleanly — Alice has manager_id = NULL)
]

employees_schema = StructType([
    StructField("employee_id", IntegerType(),      False),
    StructField("name",        StringType(),       False),
    StructField("salary",      DecimalType(10, 2), False),
    StructField("manager_id",  IntegerType(),      True),   # nullable!
])

employees = spark.createDataFrame(employees_data, employees_schema)
employees.createOrReplaceTempView("employees")
spark.sql("""
    SELECT e.employee_id AS employee_id, e.name AS employee_name, e.salary AS employee_salary, 
          m.employee_id AS manager_id, m.name AS manager_name, m.salary AS manager_salary, (e.salary - m.salary) AS salary_diff
    FROM employees e INNER JOIN employees m 
    ON e.manager_id = m.employee_id
    WHERE e.salary > m.salary
    ORDER BY salary_diff DESC
""").show()

+-----------+-------------+---------------+----------+------------+--------------+-----------+
|employee_id|employee_name|employee_salary|manager_id|manager_name|manager_salary|salary_diff|
+-----------+-------------+---------------+----------+------------+--------------+-----------+
|          9|   Ivy Jensen|      200000.00|         5|   Eve Patel|     145000.00|   55000.00|
|          7|     Grace Wu|      175000.00|         4|    Dave Kim|     150000.00|   25000.00|
|         11|   Kate Davis|      170000.00|         6|   Frank Liu|     160000.00|   10000.00|
+-----------+-------------+---------------+----------+------------+--------------+-----------+



In [10]:
"""
Same employees table — no need to reload. Now HR wants a different report: for each employee, show their direct manager AND their manager's manager (two levels up the chain).
"""

spark.sql("SELECT * FROM employees").show()

spark.sql("""
    SELECT e.employee_id AS employee_id, e.name AS employee_name, m.name AS direct_manager_name, mm.name AS second_level_manager_name 
    FROM employees e INNER JOIN employees m
    ON e.manager_id = m.employee_id
    INNER JOIN employees mm
    ON m.manager_id = mm.employee_id 
    ORDER BY employee_id
""").show()

+-----------+------------+---------+----------+
|employee_id|        name|   salary|manager_id|
+-----------+------------+---------+----------+
|          1|  Alice Chen|500000.00|      NULL|
|          2|Bob Martinez|300000.00|         1|
|          3| Carol Singh|280000.00|         1|
|          4|    Dave Kim|150000.00|         2|
|          5|   Eve Patel|145000.00|         2|
|          6|   Frank Liu|160000.00|         3|
|          7|    Grace Wu|175000.00|         4|
|          8| Henry Adams| 95000.00|         4|
|          9|  Ivy Jensen|200000.00|         5|
|         10|  Jack Brown| 80000.00|         5|
|         11|  Kate Davis|170000.00|         6|
|         12| Liam Foster| 85000.00|         6|
+-----------+------------+---------+----------+



+-----------+-------------+-------------------+-------------------------+
|employee_id|employee_name|direct_manager_name|second_level_manager_name|
+-----------+-------------+-------------------+-------------------------+
|          4|     Dave Kim|       Bob Martinez|               Alice Chen|
|          5|    Eve Patel|       Bob Martinez|               Alice Chen|
|          6|    Frank Liu|        Carol Singh|               Alice Chen|
|          7|     Grace Wu|           Dave Kim|             Bob Martinez|
|          8|  Henry Adams|           Dave Kim|             Bob Martinez|
|          9|   Ivy Jensen|          Eve Patel|             Bob Martinez|
|         10|   Jack Brown|          Eve Patel|             Bob Martinez|
|         11|   Kate Davis|          Frank Liu|              Carol Singh|
|         12|  Liam Foster|          Frank Liu|              Carol Singh|
+-----------+-------------+-------------------+-------------------------+



In [11]:
"""
Using the same dim_customer table (no reload needed), produce a "change log": every time a customer's city or segment changed, output a row showing what changed and when.
"""

from datetime import date
from decimal import Decimal
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DecimalType, BooleanType

dim_customer_data = [
    # Alice (100): moved from Seattle to Chicago + segment upgrade
    (100, "Alice Chen",   "Seattle", "RETAIL",          date(2020,1,1),  date(2022,12,31), False),
    (100, "Alice Chen",   "Chicago", "PREMIUM",         date(2023,1,1),  date(9999,12,31), True),

    # Bob (101): never changed
    (101, "Bob Martinez", "Boston",  "PRIVATE_BANKING", date(2019,6,15), date(9999,12,31), True),

    # Carol (102): three versions — segment upgrade, then move
    (102, "Carol Singh",  "New York","RETAIL",          date(2021,3,1),  date(2022,8,31),  False),
    (102, "Carol Singh",  "New York","PREMIUM",         date(2022,9,1),  date(2023,6,30),  False),
    (102, "Carol Singh",  "Miami",   "PREMIUM",         date(2023,7,1),  date(9999,12,31), True),
]

dim_customer_schema = StructType([
    StructField("customer_id",    IntegerType(), False),
    StructField("name",           StringType(),  False),
    StructField("city",           StringType(),  False),
    StructField("segment",        StringType(),  False),
    StructField("effective_date", DateType(),    False),
    StructField("end_date",       DateType(),    False),
    StructField("is_current",     BooleanType(), False),
])

fact_transactions_data = [
    # Alice
    (1, 100, date(2022, 6, 15),  Decimal( "500.00"), "Amazon"),    # Seattle/RETAIL
    (2, 100, date(2023, 3, 20),  Decimal("1000.00"), "Apple"),     # Chicago/PREMIUM
    (3, 100, date(2022, 12, 31), Decimal( "200.00"), "Costco"),    # boundary: last day Seattle
    (4, 100, date(2023, 1, 1),   Decimal( "300.00"), "Walmart"),   # boundary: first day Chicago

    # Bob
    (5, 101, date(2021, 11, 10), Decimal("5000.00"), "Tesla"),     # Boston/PRIVATE_BANKING

    # Carol
    (6, 102, date(2022, 5, 22),  Decimal( "150.00"), "CVS"),       # NY/RETAIL
    (7, 102, date(2022, 9, 1),   Decimal( "800.00"), "Macys"),     # boundary: first day PREMIUM
    (8, 102, date(2023, 7, 1),   Decimal( "400.00"), "BestBuy"),   # boundary: first day Miami
    (9, 102, date(2023, 8, 15),  Decimal( "600.00"), "GAP"),       # Miami/PREMIUM
]

fact_transactions_schema = StructType([
    StructField("txn_id",      IntegerType(),     False),
    StructField("customer_id", IntegerType(),     False),
    StructField("txn_date",    DateType(),        False),
    StructField("amount",      DecimalType(10,2), False),
    StructField("merchant",    StringType(),      False),
])

dim_customer = spark.createDataFrame(dim_customer_data, dim_customer_schema)
fact_transactions = spark.createDataFrame(fact_transactions_data, fact_transactions_schema)
dim_customer.createOrReplaceTempView("dim_customer")
fact_transactions.createOrReplaceTempView("fact_transactions")
spark.sql("SELECT * FROM dim_customer").show()
spark.sql("SELECT * FROM fact_transactions").show()
spark.sql("""
    SELECT ft.txn_id, ft.customer_id, ft.txn_date, ft.amount, dc.name AS customer_name, dc.city, dc.segment
    FROM fact_transactions ft 
    JOIN dim_customer dc ON ft.customer_id = dc.customer_id AND ft.txn_date BETWEEN dc.effective_date AND dc.end_date
    ORDER BY txn_id
""").show()

+-----------+------------+--------+---------------+--------------+----------+----------+
|customer_id|        name|    city|        segment|effective_date|  end_date|is_current|
+-----------+------------+--------+---------------+--------------+----------+----------+
|        100|  Alice Chen| Seattle|         RETAIL|    2020-01-01|2022-12-31|     false|
|        100|  Alice Chen| Chicago|        PREMIUM|    2023-01-01|9999-12-31|      true|
|        101|Bob Martinez|  Boston|PRIVATE_BANKING|    2019-06-15|9999-12-31|      true|
|        102| Carol Singh|New York|         RETAIL|    2021-03-01|2022-08-31|     false|
|        102| Carol Singh|New York|        PREMIUM|    2022-09-01|2023-06-30|     false|
|        102| Carol Singh|   Miami|        PREMIUM|    2023-07-01|9999-12-31|      true|
+-----------+------------+--------+---------------+--------------+----------+----------+

+------+-----------+----------+-------+--------+
|txn_id|customer_id|  txn_date| amount|merchant|
+------+---

+------+-----------+----------+-------+-------------+--------+---------------+
|txn_id|customer_id|  txn_date| amount|customer_name|    city|        segment|
+------+-----------+----------+-------+-------------+--------+---------------+
|     1|        100|2022-06-15| 500.00|   Alice Chen| Seattle|         RETAIL|
|     2|        100|2023-03-20|1000.00|   Alice Chen| Chicago|        PREMIUM|
|     3|        100|2022-12-31| 200.00|   Alice Chen| Seattle|         RETAIL|
|     4|        100|2023-01-01| 300.00|   Alice Chen| Chicago|        PREMIUM|
|     5|        101|2021-11-10|5000.00| Bob Martinez|  Boston|PRIVATE_BANKING|
|     6|        102|2022-05-22| 150.00|  Carol Singh|New York|         RETAIL|
|     7|        102|2022-09-01| 800.00|  Carol Singh|New York|        PREMIUM|
|     8|        102|2023-07-01| 400.00|  Carol Singh|   Miami|        PREMIUM|
|     9|        102|2023-08-15| 600.00|  Carol Singh|   Miami|        PREMIUM|
+------+-----------+----------+-------+-------------

In [12]:
spark.sql("SELECT * FROM dim_customer").show()
spark.sql("""
    WITH lagged AS(
        SELECT customer_id, name, effective_date AS change_date, city, LAG(city) OVER (PARTITION BY customer_id ORDER BY effective_date) AS previous_city, 
          segment, LAG(segment) OVER (PARTITION BY customer_id ORDER BY effective_date) AS pervious_segment,
          CASE
            WHEN (city <> LAG(city) OVER (PARTITION BY customer_id ORDER BY effective_date)) AND (segment <> LAG(segment) OVER (PARTITION BY customer_id ORDER BY effective_date)) THEN 'BOTH'
            WHEN city <> LAG(city) OVER (PARTITION BY customer_id ORDER BY effective_date) THEN 'CITY'
            WHEN segment <> LAG(segment) OVER (PARTITION BY customer_id ORDER BY effective_date) THEN 'SEGMENT'
            ELSE 'NOTHING'
          END AS what_changed
        FROM dim_customer
    )
    SELECT * FROM lagged WHERE what_changed <> 'NOTHING' ORDER BY customer_id, change_date
""").show()

+-----------+------------+--------+---------------+--------------+----------+----------+
|customer_id|        name|    city|        segment|effective_date|  end_date|is_current|
+-----------+------------+--------+---------------+--------------+----------+----------+
|        100|  Alice Chen| Seattle|         RETAIL|    2020-01-01|2022-12-31|     false|
|        100|  Alice Chen| Chicago|        PREMIUM|    2023-01-01|9999-12-31|      true|
|        101|Bob Martinez|  Boston|PRIVATE_BANKING|    2019-06-15|9999-12-31|      true|
|        102| Carol Singh|New York|         RETAIL|    2021-03-01|2022-08-31|     false|
|        102| Carol Singh|New York|        PREMIUM|    2022-09-01|2023-06-30|     false|
|        102| Carol Singh|   Miami|        PREMIUM|    2023-07-01|9999-12-31|      true|
+-----------+------------+--------+---------------+--------------+----------+----------+



+-----------+-----------+-----------+--------+-------------+-------+----------------+------------+
|customer_id|       name|change_date|    city|previous_city|segment|pervious_segment|what_changed|
+-----------+-----------+-----------+--------+-------------+-------+----------------+------------+
|        100| Alice Chen| 2023-01-01| Chicago|      Seattle|PREMIUM|          RETAIL|        BOTH|
|        102|Carol Singh| 2022-09-01|New York|     New York|PREMIUM|          RETAIL|     SEGMENT|
|        102|Carol Singh| 2023-07-01|   Miami|     New York|PREMIUM|         PREMIUM|        CITY|
+-----------+-----------+-----------+--------+-------------+-------+----------------+------------+



In [13]:
"""
A bank teller logs the customer accounts they helped during a shift, in order. To assess workload variety, the analytics team wants the longest contiguous stretch where every account is unique (no account appears twice within that stretch).
Given a list of account IDs (integers), return the length of that longest stretch.
"""

from typing import List
from collections import defaultdict

def longest_unique_streak(visits: List[int]) -> int:
    # your code here
    # res_list = list()
    # max_len = 0
    # for i in range(len(visits)):
    #     res_list.append(visits[i])
    #     if len(res_list) > len(set(res_list)):
    #         res_list = res_list[1:]
    #     else:
    #         max_len = max(max_len, len(res_list))
    # return max_len
    left, max_len = 0, 0
    seen = set()
    for right in range(len(visits)):
        while visits[right] in seen:
            seen.remove(visits[left])
            left += 1
        seen.add(visits[right])
        max_len = max(max_len, right - left + 1)
    return max_len
    pass

# Tests
print(longest_unique_streak([1, 2, 1, 3, 4, 1, 2, 5]))  # 5
print(longest_unique_streak([7, 7, 7, 7]))               # 1
print(longest_unique_streak([10, 20, 30, 40, 50]))       # 5
print(longest_unique_streak([]))                          # 0

5
1
5
0


In [14]:
"""
A bank's product team is analyzing how quickly customers reach a spending target. Given a list of daily spending amounts (positive integers, in chronological order) and a target dollar amount, return the minimum number of consecutive days whose total spending is at least the target.
If no contiguous window of days reaches the target, return 0.
"""

from typing import List

def min_days_to_target(amounts: List[int], target: int) -> int:
    # your code here
    min_len = 100000
    left = 0
    total_sum = 0
    
    for right in range(len(amounts)):
        total_sum += amounts[right]
        while total_sum >= target:
            min_len = min(min_len, right - left + 1)
            total_sum -= amounts[left]
            left += 1
        # print(total_sum, min_len, amounts[left : right + 1])
    if min_len == 100000:
        return 0
    return min_len
    pass

# Tests
print(min_days_to_target([2, 3, 1, 2, 4, 3], 7))    # 2
print(min_days_to_target([1, 1, 1, 1, 1], 10))      # 0
print(min_days_to_target([1, 4, 4], 4))             # 1
print(min_days_to_target([10, 2, 3], 6))            # 1

2
0
1
1


In [15]:
"""
A bank's trading desk has a list of daily stock prices in chronological order. For each day, they want to know how many days they'd need to wait until the price is strictly higher than that day's price. 
If there's no such future day, return 0 for that day
"""

from typing import List

def days_until_higher(prices: List[int]) -> List[int]:
    # your code here
    res, stack = [0] * len(prices), list()
    for i, price in enumerate(prices):
        while stack and prices[stack[-1]] < price:
            j = stack.pop()
            res[j] = i - j
        stack.append(i)
    return res
    pass

# Tests
print(days_until_higher([73, 74, 75, 71, 69, 72, 76, 73]))  # [1, 1, 4, 2, 1, 1, 0, 0]
print(days_until_higher([50, 50, 50]))                       # [0, 0, 0]
print(days_until_higher([10, 20]))                           # [1, 0]

[1, 1, 4, 2, 1, 1, 0, 0]
[0, 0, 0]
[1, 0]


In [16]:
"""
A bank's IT team schedules server maintenance windows. Some windows overlap or touch — these should be merged into single consolidated windows for accurate downtime reporting.
Given a list of [start, end] intervals (in minutes since midnight), merge all overlapping or adjacent windows and return the consolidated list, sorted by start time.
"""

from typing import List

def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    # your code here
    res = list()
    intervals = sorted(intervals, key = lambda x: x[0])
    i = 0
    while i < len(intervals):
        if res and res[-1][-1] >= intervals[i][0]:
            res[-1] = [res[-1][0], max(intervals[i][-1], res[-1][-1])]
        else:
            res.append(intervals[i])
        i += 1
        # print(res)
    return res
    pass

# Tests
print(merge_intervals([[1,3], [2,6], [8,10], [15,18]]))    # [[1,6], [8,10], [15,18]]
print(merge_intervals([[1,4], [4,5]]))                      # [[1,5]]
print(merge_intervals([[1,4], [2,3]]))                      # [[1,4]]
print(merge_intervals([[5,8], [1,3], [7,10], [2,4]]))       # [[1,4], [5,10]]

[[1, 6], [8, 10], [15, 18]]
[[1, 5]]
[[1, 4]]
[[1, 4], [5, 10]]


In [17]:
"""
You're given an array of daily stock prices for a bank in chronological order. Choose a single day to buy and a later day to sell to maximize profit. Return the maximum profit. If no profit is possible, return 0.
"""

from typing import List

def max_profit(prices: List[int]) -> int:
    # your code here
    # i, j = 0, 1
    # max_profit = 0
    # while i < len(prices) and j < len(prices):
    #     max_profit = max(max_profit, prices[j] - prices[i])
    #     if prices[i] > prices[j]:
    #         i += 1
    #     j += 1
    # return max_profit
    min_price = float('inf')
    max_profit = 0
    for price in prices:
        if price < min_price:
            min_price = price
        else:
            max_profit = max(max_profit, price - min_price)
    return max_profit
    pass

# Tests
print(max_profit([7, 1, 5, 3, 6, 4]))    # 5  (buy at 1, sell at 6)
print(max_profit([7, 6, 4, 3, 1]))       # 0  (prices only fall)
print(max_profit([2, 4, 1]))             # 2  (buy at 2, sell at 4)
print(max_profit([5]))                   # 0  (only one day)

5
0
2
0


In [18]:
from typing import List

def two_sum_sorted(amounts: List[int], target: int) -> List[int]:
    # your code here
    # seen = set()
    # for i in range(len(amounts)):
    #     diff = target - amounts[i]
    #     if amounts[i] in seen:
    #         return [amounts.index(diff) + 1, i + 1]
    #     else:
    #         seen.add(diff)
    #     # print(seen, diff)
    left, right = 0, len(amounts) - 1
    while left < right:
        s = amounts[left] + amounts[right]
        if s == target:
            return[left + 1, right + 1]
        elif s < target:
            left += 1
        else:
            right -= 1
    

    pass

# Tests
print(two_sum_sorted([2, 7, 11, 15], 9))         # [1, 2]   (2 + 7)
print(two_sum_sorted([2, 3, 4], 6))              # [1, 3]   (2 + 4)
print(two_sum_sorted([-1, 0], -1))               # [1, 2]   (-1 + 0)
print(two_sum_sorted([1, 2, 3, 4, 4], 8))        # [4, 5]   (4 + 4)

[1, 2]
[1, 3]
[1, 2]
[4, 5]


In [19]:
from datetime import datetime
from decimal import Decimal
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, DecimalType

txn_data = [
    # Account 100: 5 txns, most recent ABOVE avg
    (1,  100, datetime(2024, 1, 10,  9, 0, 0), Decimal("100.00")),
    (2,  100, datetime(2024, 1, 25, 14, 0, 0), Decimal("200.00")),
    (3,  100, datetime(2024, 2,  8, 10, 0, 0), Decimal("150.00")),
    (4,  100, datetime(2024, 2, 20, 11, 0, 0), Decimal("250.00")),
    (5,  100, datetime(2024, 3, 15, 16, 0, 0), Decimal("300.00")),  # most recent

    # Account 200: 3 txns, most recent BELOW avg
    (6,  200, datetime(2024, 1,  5,  9, 0, 0), Decimal("500.00")),
    (7,  200, datetime(2024, 2, 18, 13, 0, 0), Decimal("300.00")),
    (8,  200, datetime(2024, 3, 22, 10, 0, 0), Decimal("100.00")),  # most recent

    # Account 300: only 1 txn — pct_diff should be 0
    (9,  300, datetime(2024, 2, 14, 12, 0, 0), Decimal("150.00")),

    # Account 400: TIE on txn_date — break by txn_id DESC
    (10, 400, datetime(2024, 3,  1,  9, 0, 0), Decimal("100.00")),
    (50, 400, datetime(2024, 3, 15, 10, 0, 0), Decimal("200.00")),
    (51, 400, datetime(2024, 3, 15, 10, 0, 0), Decimal("300.00")),  # ties on date with txn 50; higher txn_id wins

    # Account 500: mix of positive and negative amounts
    (12, 500, datetime(2024, 1, 10,  8, 0, 0), Decimal( "100.00")),
    (13, 500, datetime(2024, 2,  5,  9, 0, 0), Decimal( "-50.00")),
    (14, 500, datetime(2024, 3,  8, 10, 0, 0), Decimal( "200.00")),  # most recent
]

txn_schema = StructType([
    StructField("txn_id",     IntegerType(),     False),
    StructField("account_id", IntegerType(),     False),
    StructField("txn_date",   TimestampType(),   False),
    StructField("amount",     DecimalType(10,2), False),
])

transactions = spark.createDataFrame(txn_data, txn_schema)
transactions.createOrReplaceTempView("transactions")
transactions.show()
spark.sql("""
    WITH first_cte AS(
        SELECT account_id, txn_id, txn_date, amount, DENSE_RANK() OVER (PARTITION BY account_id ORDER BY txn_date DESC, txn_id DESC) AS rank,
          COUNT(*) OVER (PARTITION BY account_id) AS total_txn_count,
          AVG(amount) OVER (PARTITION BY account_id) AS avg_amount
        FROM transactions      
    )
    SELECT f.account_id, f.txn_id AS most_recent_txn_id, f.txn_date AS most_recent_txn_date, f.amount AS most_recenet_txn_amount, f.total_txn_count, f.avg_amount,
          ROUND(((f.amount - f.avg_amount)/f.avg_amount) * 100, 2) AS pct_diff_from_avg
    FROM first_cte f
    WHERE f.rank = 1
    ORDER BY account_id
""").show()

+------+----------+-------------------+------+
|txn_id|account_id|           txn_date|amount|
+------+----------+-------------------+------+
|     1|       100|2024-01-10 09:00:00|100.00|
|     2|       100|2024-01-25 14:00:00|200.00|
|     3|       100|2024-02-08 10:00:00|150.00|
|     4|       100|2024-02-20 11:00:00|250.00|
|     5|       100|2024-03-15 16:00:00|300.00|
|     6|       200|2024-01-05 09:00:00|500.00|
|     7|       200|2024-02-18 13:00:00|300.00|
|     8|       200|2024-03-22 10:00:00|100.00|
|     9|       300|2024-02-14 12:00:00|150.00|
|    10|       400|2024-03-01 09:00:00|100.00|
|    50|       400|2024-03-15 10:00:00|200.00|
|    51|       400|2024-03-15 10:00:00|300.00|
|    12|       500|2024-01-10 08:00:00|100.00|
|    13|       500|2024-02-05 09:00:00|-50.00|
|    14|       500|2024-03-08 10:00:00|200.00|
+------+----------+-------------------+------+

+----------+------------------+--------------------+-----------------------+---------------+----------+---

In [20]:
from typing import List, Tuple

def min_workstations(sessions: List[Tuple[int, int]]) -> int:
    # sessions: list of (start, end) tuples (e.g., minutes since midnight)
    # return: int — minimum number of workstations needed
    events = list()
    for start, end in sessions:
        events.append((start, +1))
        events.append((end, -1))
    events = sorted(events)
    current_active, max_active = 0, 0
    for time, delta in events:
        current_active += delta
        max_active = max(max_active, current_active)
    return max_active
    
    pass
    

# Tests
print(min_workstations([(0, 30), (5, 10), (15, 20)]))      # 2
print(min_workstations([(7, 10), (2, 4)]))                  # 1 (don't overlap)
print(min_workstations([(0, 30), (5, 30), (10, 30)]))       # 3 (all overlap)
print(min_workstations([(0, 5), (5, 10), (10, 15)]))        # 1 (touch but don't overlap)
print(min_workstations([]))                                  # 0

2
1
3
1
0


In [21]:
"""
For each branch, show month-by-month deposit totals AND a running cumulative total. Pure muscle memory.
"""

from datetime import date
from decimal import Decimal
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DecimalType

deposits_data = [
    (1, date(2024, 1, 5),  Decimal("1000.00")),
    (1, date(2024, 1, 20), Decimal("500.00")),
    (1, date(2024, 2, 8),  Decimal("1200.00")),
    (1, date(2024, 3, 3),  Decimal("800.00")),

    (2, date(2024, 1, 10), Decimal("2000.00")),
    (2, date(2024, 2, 14), Decimal("1500.00")),
    (2, date(2024, 3, 5),  Decimal("2500.00")),

    (3, date(2024, 2, 18), Decimal("700.00")),
    (3, date(2024, 3, 12), Decimal("900.00")),
]

deposits_schema = StructType([
    StructField("branch_id",    IntegerType(),     False),
    StructField("deposit_date", DateType(),        False),
    StructField("amount",       DecimalType(10,2), False),
])

deposits = spark.createDataFrame(deposits_data, deposits_schema)
deposits.createOrReplaceTempView("deposits")
spark.sql("SELECT * FROM deposits").show()
spark.sql("""
    WITH first_cte AS (
        SELECT branch_id, trunc(deposit_date, 'MM') AS month, SUM(amount) AS total_amount
        FROM deposits
        GROUP BY branch_id, month      
    )
    SELECT branch_id, month, total_amount, SUM(total_amount) OVER (PARTITION BY branch_id ORDER BY month) AS cumulative_total
    FROM first_cte
    ORDER BY branch_id, month
""").show()

+---------+------------+-------+
|branch_id|deposit_date| amount|
+---------+------------+-------+
|        1|  2024-01-05|1000.00|
|        1|  2024-01-20| 500.00|
|        1|  2024-02-08|1200.00|
|        1|  2024-03-03| 800.00|
|        2|  2024-01-10|2000.00|
|        2|  2024-02-14|1500.00|
|        2|  2024-03-05|2500.00|
|        3|  2024-02-18| 700.00|
|        3|  2024-03-12| 900.00|
+---------+------------+-------+



+---------+----------+------------+----------------+
|branch_id|     month|total_amount|cumulative_total|
+---------+----------+------------+----------------+
|        1|2024-01-01|     1500.00|         1500.00|
|        1|2024-02-01|     1200.00|         2700.00|
|        1|2024-03-01|      800.00|         3500.00|
|        2|2024-01-01|     2000.00|         2000.00|
|        2|2024-02-01|     1500.00|         3500.00|
|        2|2024-03-01|     2500.00|         6000.00|
|        3|2024-02-01|      700.00|          700.00|
|        3|2024-03-01|      900.00|         1600.00|
+---------+----------+------------+----------------+



In [22]:
"""
Given a list of transactions, return a dict mapping each category → the customer_id who spent the most in that category.
"""

from typing import List, Dict
from collections import defaultdict

def top_spender_per_category(transactions: List[dict]) -> Dict[str, int]:
    # your code here
    spends = defaultdict(int)
    for transaction in transactions:
        spends[(transaction['customer_id'], transaction['category'])] += transaction['amount']
    # spends = dict(sorted(spends.items(), key = lambda x: x[1], reverse = True))
    print(spends)
    res = dict()
    for (customer, category), amount in spends.items():
        if category not in res or amount > res[category][1]:
            res[category] = (customer, amount)
    print(res)
    return {cat: cust for cat, (cust, _) in res.items()}

    pass

# Test
transactions = [
    {"customer_id": 101, "category": "FOOD",   "amount": 50},
    {"customer_id": 102, "category": "FOOD",   "amount": 80},
    {"customer_id": 101, "category": "FOOD",   "amount": 40},   # 101 total FOOD = 90
    {"customer_id": 103, "category": "TRAVEL", "amount": 500},
    {"customer_id": 102, "category": "TRAVEL", "amount": 300},
    {"customer_id": 101, "category": "RETAIL", "amount": 200},
]

print(top_spender_per_category(transactions))
# Expected: {'FOOD': 101, 'TRAVEL': 103, 'RETAIL': 101}
# (101 has 90 in FOOD vs 102's 80; 103 wins TRAVEL; 101 only one in RETAIL)

defaultdict(<class 'int'>, {(101, 'FOOD'): 90, (102, 'FOOD'): 80, (103, 'TRAVEL'): 500, (102, 'TRAVEL'): 300, (101, 'RETAIL'): 200})
{'FOOD': (101, 90), 'TRAVEL': (103, 500), 'RETAIL': (101, 200)}
{'FOOD': 101, 'TRAVEL': 103, 'RETAIL': 101}


In [23]:
"""
A bank's marketing team wants to identify VIP customers for an exclusive outreach campaign. A customer qualifies as VIP if all these are true:

They're in the top 25% by total spend among all customers
They have made at least 3 transactions
Their average transaction amount is at least $100
"""
from pyspark.sql.types import StructField, StructType, IntegerType, FloatType, DateType, StringType

txn_schema = StructType([
    StructField("txn_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("txn_date", DateType()),
    StructField("amount", FloatType()),
    StructField("merchant", StringType())
])

txn_df = spark.read.format("csv")\
                .schema(txn_schema)\
                .option("header", True)\
                .load(path = "/workspaces/pyspark_udemy_codespace/data/transactions.csv")
txn_df.show()
txn_df.createOrReplaceTempView("transactions")
spark.sql("""
    WITH grouped AS(
        SELECT customer_id, COUNT(*) AS txn_count, SUM(amount) AS total_spend, AVG(amount) AS avg_amount
        FROM transactions
        GROUP BY customer_id 
    ),
    ranked AS(
        SELECT customer_id, total_spend, txn_count, avg_amount, RANK() OVER (ORDER BY total_spend) AS spend_rank,
        ((RANK() OVER (ORDER BY total_spend DESC) - 1) / ((SELECT COUNT(DISTINCT customer_id) FROM transactions) - 1)) AS spend_percentile
        FROM grouped
    )
        SELECT customer_id, total_spend, txn_count, avg_amount, spend_rank, spend_percentile
        FROM ranked
        WHERE spend_percentile <= 0.25
        AND txn_count >= 3
        AND avg_amount >= 100
        ORDER BY total_spend DESC
""").show()

+------+-----------+----------+------+-----------+
|txn_id|customer_id|  txn_date|amount|   merchant|
+------+-----------+----------+------+-----------+
|     1|        101|2024-01-05| 250.0|Whole_Foods|
|     2|        102|2024-01-08|  75.0|  Starbucks|
|     3|        101|2024-01-15| 180.0|     Amazon|
|     4|        103|2024-01-20| 800.0|      Apple|
|     5|        102|2024-01-22|  55.0|     Target|
|     6|        101|2024-01-28| 220.0|   Best_Buy|
|     7|        104|2024-02-02|  30.0|        CVS|
|     8|        105|2024-02-05|1500.0|    Tiffany|
|     9|        101|2024-02-10| 300.0|      Apple|
|    10|        103|2024-02-12| 650.0|   Best_Buy|
|    11|        102|2024-02-18|  40.0|  Starbucks|
|    12|        106|2024-02-20|2200.0|      Rolex|
|    13|        104|2024-02-22|  25.0|        CVS|
|    14|        101|2024-02-28| 420.0|Whole_Foods|
|    15|        105|2024-03-03| 950.0|      Apple|
|    16|        103|2024-03-08| 300.0|     Amazon|
|    17|        102|2024-03-12|

26/07/03 11:50:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/03 11:50:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/03 11:50:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/03 11:50:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/03 11:50:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/03 11:50:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/03 1

+-----------+-----------+---------+------------------+----------+-------------------+
|customer_id|total_spend|txn_count|        avg_amount|spend_rank|   spend_percentile|
+-----------+-----------+---------+------------------+----------+-------------------+
|        105|     3175.0|        3|1058.3333333333333|         6|0.16666666666666666|
+-----------+-----------+---------+------------------+----------+-------------------+



In [ ]:
sales_schema = StructType([
    StructField("sale_id", IntegerType()),
    StructField("salesperson", StringType()),
    StructField("region", StringType()),
    StructField("product", StringType()),
    StructField("amount", IntegerType()),
    StructField("sale_date", DateType())
])

sales_df = spark.read.format("json")\
                     .schema(sales_schema)\
                     .load(path = "/workspaces/pyspark_udemy_codespace/data/sales.json")

sales_df.show()
sales_df.printSchema()

+-------+-----------+------+-------+------+----------+
|sale_id|salesperson|region|product|amount| sale_date|
+-------+-----------+------+-------+------+----------+
|      1|      Alice| North| Laptop|  1200|2024-01-05|
|      2|        Bob| South|  Phone|   800|2024-01-08|
|      3|      Alice| North|  Phone|   750|2024-01-15|
|      4|      Carol|  East| Laptop|  1500|2024-01-20|
|      5|        Bob| South| Laptop|  1300|2024-02-02|
|      6|      Alice| North| Tablet|   500|2024-02-10|
|      7|       Dave|  West|  Phone|   900|2024-02-14|
|      8|      Carol|  East| Tablet|   600|2024-02-20|
|      9|        Bob| South| Tablet|   450|2024-03-05|
|     10|      Alice| North| Laptop|  1400|2024-03-12|
|     11|       Dave|  West| Laptop|  1600|2024-03-18|
|     12|      Carol|  East|  Phone|   850|2024-03-25|
+-------+-----------+------+-------+------+----------+

root
 |-- sale_id: integer (nullable = true)
 |-- salesperson: string (nullable = true)
 |-- region: string (nullable =

In [37]:
"""
🟢 Q1 — Basic aggregation
For each region, find the total revenue and count of sales. Order by revenue descending.
"""
from pyspark.sql.functions import sum, col, count

q1_df = sales_df.groupBy(col("region"))\
                .agg(sum(col("amount")).alias("total_revenue"), count(col("sale_id")).alias("total_sales"))\
                .select(col("region"), col("total_revenue"), col("total_sales"))\
                .orderBy(col("total_revenue").desc())
q1_df.show()

+------+-------------+-----------+
|region|total_revenue|total_sales|
+------+-------------+-----------+
| North|         3850|          4|
|  East|         2950|          3|
| South|         2550|          3|
|  West|         2500|          2|
+------+-------------+-----------+

